In [1]:
import os, json, time, pickle
import numpy as np
import torch

os.environ["OMP_NUM_THREADS"] = "4"     
torch.set_num_threads(4)
DEVICE = "cpu"
torch.set_default_dtype(torch.float32)
ROOT   = os.path.abspath(".")
CKPT   = os.path.join(ROOT, "checkpoints")   # models, histories, ground truth
OUT    = os.path.join(ROOT, "outputs", "xai")# XAI figures + manifest
for d in (CKPT, OUT):
    os.makedirs(d, exist_ok=True)
print("checkpoints ->", CKPT)
print("xai outputs ->", OUT)

def save_pickle(obj, name):
    p = os.path.join(CKPT, name)
    with open(p, "wb") as f: pickle.dump(obj, f)
    print(f"  saved {name}")
    return p

def load_pickle(name):
    with open(os.path.join(CKPT, name), "rb") as f:
        return pickle.load(f)

def exists(name):
    return os.path.exists(os.path.join(CKPT, name))

def stamp(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

checkpoints -> d:\PINN\Burgers_1d\checkpoints
xai outputs -> d:\PINN\Burgers_1d\outputs\xai


In [2]:
import importlib
required = ["config", "models", "physics", "train", "train_qapinn",
            "ground_truth", "evaluate", "solver_spectral", "solver_fdm", "solver_fem"]
for m in required:
    importlib.import_module(m)
    print("  ok:", m)

import xai
from xai.adapter import QuantumProbe, TorchModelAdapter
from xai import report, layer2, layer3, domain, scaling
print("  ok: xai", xai.__version__)
os.makedirs("outputs", exist_ok=True)
stamp("Cell 1 done — all imports resolve")

  ok: config
  ok: models
  ok: physics
  ok: train
  ok: train_qapinn
  ok: ground_truth
  ok: evaluate
  ok: solver_spectral
  ok: solver_fdm
  ok: solver_fem
  ok: xai 1.0.0
[08:51:40] Cell 1 done — all imports resolve


In [3]:
from ground_truth import GroundTruth, cross_validate

GT_CACHE = "ground_truth.pkl"
if exists(GT_CACHE):
    stamp("loading cached ground truth")
    gts = load_pickle(GT_CACHE)
else:
    stamp("solving spectral / fdm / fem references (one-time)")
    gts = {m: GroundTruth(m) for m in ("spectral", "fdm", "fem")}
    save_pickle(gts, GT_CACHE)

gt = gts["spectral"]              
import numpy as np
xq = np.linspace(-1, 1, 401)
for t0 in (0.25, 0.5, 0.75):
    ref = gt.slice(t0, xq)
    e_fdm = np.linalg.norm(gts["fdm"].slice(t0, xq) - ref) / np.linalg.norm(ref)
    e_fem = np.linalg.norm(gts["fem"].slice(t0, xq) - ref) / np.linalg.norm(ref)
    print(f"  t={t0}:  FDM vs spectral {e_fdm:.2e}   FEM vs spectral {e_fem:.2e}")

stamp("Cell 2 done — ground truth ready")

[08:51:46] loading cached ground truth
  t=0.25:  FDM vs spectral 6.22e-04   FEM vs spectral 3.04e-04
  t=0.5:  FDM vs spectral 2.17e-03   FEM vs spectral 6.74e-04
  t=0.75:  FDM vs spectral 1.82e-03   FEM vs spectral 4.75e-04
[08:51:46] Cell 2 done — ground truth ready


In [4]:
from models import ClassicalPINN
from train import train_classical
from evaluate import evaluate

CLS_W    = "classical_pinn.pt"
CLS_HIST = "classical_hist.pkl"

CLS_CFG = dict(depth=4, width=8, adam_epochs=600, lbfgs_iter=0,
               lr=2e-3, seed=0, log_every=50,
               snapshot_epochs=(0, 100, 200, 300, 400, 500))

if exists(CLS_W) and exists(CLS_HIST):
    stamp("loading trained classical PINN")
    cl = ClassicalPINN(depth=CLS_CFG["depth"], width=CLS_CFG["width"]).to(DEVICE)
    cl.load_state_dict(torch.load(os.path.join(CKPT, CLS_W), map_location=DEVICE))
    cl.eval()
    cl_hist = load_pickle(CLS_HIST)
else:
    stamp("training classical PINN (this is the long one — grab coffee)")
    t0 = time.perf_counter()
    cl, cl_hist, cl_snaps = train_classical(**CLS_CFG)
    stamp(f"classical training wall = {time.perf_counter()-t0:.1f}s")
    torch.save(cl.state_dict(), os.path.join(CKPT, CLS_W))
    save_pickle(cl_hist, CLS_HIST)
    save_pickle(cl_snaps, "classical_snaps.pkl")

# report accuracy vs spectral GT
res_cl = evaluate(cl, gt, nx=401, nt=101)
print(f"  classical  rel-L2 = {res_cl['l2_global']:.4e}   Linf = {res_cl['linf']:.3e}")
save_pickle(res_cl, "classical_eval.pkl")
stamp("Cell 3 done — classical PINN trained & saved")

[08:51:49] loading trained classical PINN
  classical  rel-L2 = 4.3086e-01   Linf = 1.060e+00
  saved classical_eval.pkl
[08:51:49] Cell 3 done — classical PINN trained & saved


In [5]:
# CELL 4 — train the QA-PINN from scratch on CPU (the quantum arm)


from models import QAPINN
from train_qapinn import train_qapinn

QA_W    = "qapinn_4q.pt"
QA_HIST = "qapinn_4q_hist.pkl"
QA_SNAP = "qapinn_4q_snaps.pkl"
QA_CFG = dict(n_qubits=3, n_layers=2, hidden=8, reupload=False,
              epochs=600, lr=2e-3, seed=0,
              n_pde=256, n_ic=128, n_bc=128, log_every=50,
              snapshot_epochs=(0, 100, 200, 300, 400, 500))

if exists(QA_W) and exists(QA_HIST) and exists(QA_SNAP):
    stamp("loading trained QA-PINN")
    qa = QAPINN(n_qubits=QA_CFG["n_qubits"], hidden=QA_CFG["hidden"],
                n_layers=QA_CFG["n_layers"], reupload=QA_CFG["reupload"]).to(DEVICE)
    qa.load_state_dict(torch.load(os.path.join(CKPT, QA_W), map_location=DEVICE))
    qa.eval()
    qa_hist  = load_pickle(QA_HIST)
    qa_snaps = load_pickle(QA_SNAP)
else:
    stamp("training QA-PINN (quantum sim on CPU — slowest cell)")
    t0 = time.perf_counter()
    qa, qa_hist, qa_snaps = train_qapinn(**QA_CFG)
    stamp(f"QA-PINN training wall = {time.perf_counter()-t0:.1f}s")
    torch.save(qa.state_dict(), os.path.join(CKPT, QA_W))
    save_pickle(qa_hist,  QA_HIST)
    save_pickle(qa_snaps, QA_SNAP)

res_qa = evaluate(qa, gt, nx=401, nt=101)
print(f"  QA-PINN   rel-L2 = {res_qa['l2_global']:.4e}   Linf = {res_qa['linf']:.3e}")
save_pickle(res_qa, "qapinn_4q_eval.pkl")
stamp("Cell 4 done — QA-PINN trained & saved")

[08:51:54] loading trained QA-PINN
  QA-PINN   rel-L2 = 5.4252e-01   Linf = 9.821e-01
  saved qapinn_4q_eval.pkl
[08:51:54] Cell 4 done — QA-PINN trained & saved


In [6]:
# CELL 5 — wire trained models into the xai module via adapters
from physics import build_batches, composite_loss
from config import NU, X_MIN, X_MAX, T_MIN, T_MAX

BOUNDS = [(X_MIN, X_MAX), (T_MIN, T_MAX)]    
probe     = QuantumProbe.from_qapinn(qa, device=DEVICE, d_in=2)
classical = TorchModelAdapter(cl, device=DEVICE, name="classical_pinn", d_in=2)

B_fixed = build_batches(n_pde=256, n_ic=128, n_bc=128, seed=123)
loss_q = lambda: composite_loss(qa, B_fixed)[0]    
loss_c = lambda: composite_loss(cl, B_fixed)[0]     


def residual_fn(adapter, X):
    Xr = X.clone().detach().requires_grad_(True)
    x, t = Xr[:, 0:1], Xr[:, 1:2]
    u  = adapter(torch.cat([x, t], 1))
    ut = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    ux = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    uxx= torch.autograd.grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    return (ut + u*ux - NU*uxx).detach().cpu().numpy()

# sanity: residual should be small on the trained QA-PINN interior
import numpy as np
Xt = torch.rand(200, 2) * torch.tensor([2.0, 1.0]) + torch.tensor([-1.0, 0.0])
print("  mean |PDE residual| (QA-PINN):", np.abs(residual_fn(probe, Xt)).mean())
stamp("Cell 5 done — adapters + closures ready")

  mean |PDE residual| (QA-PINN): 0.63172853
[08:51:59] Cell 5 done — adapters + closures ready


In [7]:
# CELL 6 — LAYER 2: quantum-layer explainability (the core battery)


L2 = {}
stamp("2.1 input sensitivity");        L2["2.1"] = layer2.input_sensitivity(probe, BOUNDS, n=1500, outdir=OUT)
stamp("2.2 measurement operators");    L2["2.2"] = layer2.measurement_operators(probe, BOUNDS, residual_fn=residual_fn, n=800, outdir=OUT)
stamp("2.4 entanglement");             L2["2.4"] = layer2.entanglement_analysis(probe, BOUNDS, residual_fn=residual_fn, n=512, outdir=OUT)
stamp("2.5 fourier spectrum");         L2["2.5"] = layer2.fourier_spectrum(probe, BOUNDS, classical_ref=classical, outdir=OUT)
stamp("2.6 expressivity");             L2["2.6"] = layer2.expressivity_analysis(probe, BOUNDS, n_states=400, outdir=OUT)
stamp("2.9 measurement distribution"); L2["2.9"] = layer2.measurement_distribution(probe, BOUNDS, n=800, outdir=OUT)
stamp("2.10 feature attribution (Q)"); L2["2.10_q"] = layer2.feature_attribution(probe, BOUNDS, outdir=OUT)
stamp("2.10 feature attribution (C)"); L2["2.10_c"] = layer2.feature_attribution(classical, BOUNDS, outdir=OUT)

# 2.8 loss landscape (filter-normalised). n=21 grid = 441 loss evals; a few min.
stamp("2.8 loss landscape (slow-ish)")
L2["2.8"] = layer2.loss_landscape(probe, lambda: float(loss_q()), span=1.0, n=21, outdir=OUT)

# headline numbers
print(f"\n  Meyer–Wallach Q            = {L2['2.4']['meyer_wallach_Q']:.3f}  ({L2['2.4']['interpretation'][:60]}...)")
print(f"  spectral centroid (Q)      = {L2['2.5']['spectral_centroid']:.2f}   reachable f ≤ {L2['2.5']['theoretical_reachable_freq']}")
print(f"  effective dimension        = {L2['2.6']['effective_dimension']:.1f} / {L2['2.6']['feature_dim']}")
print(f"  measurement entropy        = {L2['2.9']['mean_entropy']:.2f} bits")

save_pickle(L2, "layer2_results.pkl")
stamp("Cell 6 done — Layer 2 core complete (see outputs/xai/*.png)")

[08:52:03] 2.1 input sensitivity
[08:52:03] 2.2 measurement operators
[08:52:04] 2.4 entanglement
[08:52:04] 2.5 fourier spectrum
[08:52:05] 2.6 expressivity
[08:52:06] 2.9 measurement distribution
[08:52:07] 2.10 feature attribution (Q)
[08:52:07] 2.10 feature attribution (C)
[08:52:08] 2.8 loss landscape (slow-ish)

  Meyer–Wallach Q            = 0.283  (Mild entanglement: the quantum layer uses correlations modes...)
  spectral centroid (Q)      = 4.19   reachable f ≤ 1
  effective dimension        = 1.7 / 8
  measurement entropy        = 1.57 bits
  saved layer2_results.pkl
[08:52:47] Cell 6 done — Layer 2 core complete (see outputs/xai/*.png)


In [8]:
# CELL 7 — LAYER 3: optimisation geometry (classical vs quantum, side by side)

L3 = {}
stamp("Layer 3 — QA-PINN optimisation report")
L3["qa"] = layer3.optimization_report(probe, loss_q, hist=qa_hist, outdir=OUT)

stamp("Layer 3 — classical optimisation report")
L3["cl"] = layer3.optimization_report(classical, loss_c, hist=cl_hist, outdir=OUT)

for tag, r in (("QA-PINN", L3["qa"]), ("classical", L3["cl"])):
    h = r["hessian"]
    print(f"\n  {tag}:")
    print(f"    grad-norm {r['gradient']['grad_norm_mean']:.3e}  param-norm {r['gradient']['param_norm']:.3e}")
    if h:
        print(f"    λmax {h['lambda_max']:.3e}  cond {h['condition_number']:.2e}  lr-ceiling {h['lr_ceiling']:.2e}")
    if r["stability"]:
        print(f"    90% converged @ iter {r['stability']['iters_to_90pct']}  late-vol {r['stability']['late_volatility']:.2e}")

save_pickle(L3, "layer3_results.pkl")
stamp("Cell 7 done — Layer 3 complete")

[08:53:03] Layer 3 — QA-PINN optimisation report
[08:53:40] Layer 3 — classical optimisation report

  QA-PINN:
    grad-norm 4.345e+00  param-norm 7.521e+00
    λmax 4.589e+03  cond 2.04e+05  lr-ceiling 4.36e-04
    90% converged @ iter 245  late-vol 4.37e-03

  classical:
    grad-norm 5.989e+00  param-norm 6.247e+00
    λmax 4.777e+03  cond 7.53e+05  lr-ceiling 4.19e-04
    90% converged @ iter 130  late-vol 1.79e-02
  saved layer3_results.pkl
[08:53:43] Cell 7 done — Layer 3 complete


In [9]:
# CELL 8 — LAYER 2.11: quantum-state evolution across training

from models import QAPINN

# extract q_weights arrays from the saved state_dict snapshots
weight_snaps = {ep: sd["q_weights"].cpu().numpy() for ep, sd in qa_snaps.items()}
print("  snapshot epochs:", sorted(weight_snaps))

def make_probe_from_weights(w):
    m = QAPINN(n_qubits=QA_CFG["n_qubits"], n_layers=QA_CFG["n_layers"],
               reupload=QA_CFG["reupload"]).to(DEVICE)
    with torch.no_grad():
        m.q_weights.copy_(torch.tensor(w))
    return QuantumProbe.from_qapinn(m, device=DEVICE, d_in=2)

stamp("2.11 quantum-state evolution")
evo = layer2.quantum_state_evolution(make_probe_from_weights, weight_snaps,
                                     BOUNDS, outdir=OUT)
print("  fidelity between consecutive snapshots:", [round(f, 3) for f in evo["fidelity"]])
save_pickle(evo, "layer2_11_evolution.pkl")
stamp("Cell 8 done — state evolution complete")

  snapshot epochs: [0, 100, 200, 300, 400, 500]
[08:53:50] 2.11 quantum-state evolution
  fidelity between consecutive snapshots: [1.0, 0.936, 0.95, 0.976, 1.0, 1.0]
  saved layer2_11_evolution.pkl
[08:53:50] Cell 8 done — state evolution complete


In [10]:
# ============================================================================
# CELL 9 — LAYER 2.7: barren-plateau scan (no training — random inits only)
# ============================================================================
# Measures gradient variance vs #qubits and depth. Each fresh probe is a NEW
# random-init QA-PINN. Keep qubit_list modest on CPU (6 qubits = 64-dim states).

def make_fresh(nq, depth):
    m = QAPINN(n_qubits=nq, n_layers=depth, reupload=False).to(DEVICE)
    return QuantumProbe.from_qapinn(m, device=DEVICE, d_in=2)

stamp("2.7 barren-plateau scan (CPU: keep qubits ≤ 6)")
bp = layer2.barren_plateau_analysis(
    make_fresh, BOUNDS,
    qubit_list=(2, 4, 6), depth_list=(2, 4, 8),
    n_samples=40, n_points=64, outdir=OUT)
save_pickle(bp, "layer2_7_barren.pkl")
stamp("Cell 9 done — barren-plateau scan complete")

[08:53:52] 2.7 barren-plateau scan (CPU: keep qubits ≤ 6)
  saved layer2_7_barren.pkl
[08:54:16] Cell 9 done — barren-plateau scan complete


In [11]:
# ============================================================================
# CELL 10 — LAYER 2.3: circuit-depth sweep (short trainings, checkpointed)
# ============================================================================
# Trains a SMALL QA-PINN at each depth so the sweep is affordable on CPU.
# Each depth's model is cached so re-running is instant.

from train_qapinn import train_qapinn

DEPTHS = (1, 2, 4, 6, 8, 10)        

def build_train_eval(depth):
    tag = f"depth_{depth}.pkl"
    if exists(tag):
        return load_pickle(tag)
    stamp(f"  training depth={depth}")
    m, h, _ = train_qapinn(n_qubits=4, n_layers=depth, reupload=False,
                           epochs=2000, n_pde=256, n_ic=128, n_bc=64,
                           log_every=50, seed=0)
    r = evaluate(m, gt, nx=201, nt=51)
    # gradient norm at the end
    p = QuantumProbe.from_qapinn(m, device=DEVICE, d_in=2)
    gd = layer3.gradient_diagnostics(p, lambda: composite_loss(m, B_fixed)[0], n_batches=3)
    ex = layer2.expressivity_analysis(p, BOUNDS, n_states=200, plot=False, outdir=OUT)
    out = dict(train_loss=float(h["total"][-1]), rel_l2=float(r["l2_global"]),
               grad_norm=float(gd["grad_norm_mean"]),
               expressivity=float(ex["effective_dimension"]),
               runtime=float(h["wall"][-1]))
    save_pickle(out, tag)
    return out

stamp("2.3 circuit-depth sweep")
depth_res = layer2.circuit_depth_analysis(build_train_eval, depths=DEPTHS,
                                          outdir=OUT, name="qapinn_4q")
save_pickle(depth_res, "layer2_3_depth.pkl")
stamp("Cell 10 done — depth sweep complete")

[08:54:19] 2.3 circuit-depth sweep
  saved layer2_3_depth.pkl
[08:54:20] Cell 10 done — depth sweep complete


In [12]:
# CELL 11 — QUBIT SCALING: {2,4,6} qubits
def build_probe_and_metrics(nq):
    tag = f"scaling_{nq}q.pt"
    htag = f"scaling_{nq}q_hist.pkl"
    if exists(tag) and exists(htag):
        m = QAPINN(n_qubits=nq, n_layers=6, reupload=False).to(DEVICE)
        m.load_state_dict(torch.load(os.path.join(CKPT, tag), map_location=DEVICE))
        h = load_pickle(htag)
    else:
        stamp(f"  training scaling model nq={nq}")
        m, h, _ = train_qapinn(n_qubits=nq, n_layers=6, reupload=False,
                               epochs=2000, n_pde=256, n_ic=128, n_bc=128,
                               log_every=1000, seed=0)
        torch.save(m.state_dict(), os.path.join(CKPT, tag))
        save_pickle(h, htag)
    r = evaluate(m, gt, nx=201, nt=51)
    p = QuantumProbe.from_qapinn(m, device=DEVICE, d_in=2)
    return p, dict(rel_l2=float(r["l2_global"]), runtime=float(h["wall"][-1]))

stamp("qubit-scaling sweep {2,4,6}")
scale_res = scaling.qubit_scaling(build_probe_and_metrics, BOUNDS,
                                  qubit_list=(2, 4, 6, 8), n_eval=300,
                                  outdir=OUT, name="qapinn")
save_pickle(scale_res, "scaling_results.pkl")
print("\n  qubit-scaling table:")
for row in scale_res["table"]:
    print(f"    {row['n_qubits']}q: rel-L2 {row.get('rel_l2', float('nan')):.3e} "
          f"eff-dim {row.get('effective_dimension', float('nan')):.1f} "
          f"Q {row.get('meyer_wallach_Q', float('nan')):.2f} "
          f"H {row.get('measurement_entropy', float('nan')):.2f}")
stamp("Cell 11 done — qubit scaling complete")

[08:54:22] qubit-scaling sweep {2,4,6}
[08:54:26]   training scaling model nq=8
[8q L6 ru=0]     0 | 1.6107e+01 | pde 1.71e-06 ic 6.45e-01 bc 1.60e-01 | 5.7s
[8q L6 ru=0]  1000 | 3.3081e-01 | pde 3.00e-01 ic 1.40e-03 bc 1.40e-04 | 5944.0s
[8q L6 ru=0]  1999 | 3.2505e-01 | pde 2.97e-01 ic 1.29e-03 bc 1.32e-04 | 11221.8s
  saved scaling_8q_hist.pkl
  saved scaling_results.pkl

  qubit-scaling table:
    2q: rel-L2 5.188e-01 eff-dim 1.4 Q 0.34 H 1.58
    4q: rel-L2 5.215e-01 eff-dim 1.9 Q 0.72 H 3.53
    6q: rel-L2 5.154e-01 eff-dim 2.3 Q 0.91 H 5.23
    8q: rel-L2 5.057e-01 eff-dim 2.6 Q 0.97 H 7.15
[12:01:43] Cell 11 done — qubit scaling complete


In [13]:
# ============================================================================
# CELL 12 — LAYER 4: domain generalisation / extrapolation
# ============================================================================
# Extends t beyond the training window [0,1] and measures where each model
# breaks down against the spectral ground truth.
# NOTE: the cached GT was solved to t_max=1.0. To test t>1 honestly, re-solve
# the spectral reference on the extended window first.

from ground_truth import GroundTruth

GT_EXT = "ground_truth_ext.pkl"
if exists(GT_EXT):
    gt_ext = load_pickle(GT_EXT)
else:
    stamp("solving spectral GT on extended window t∈[0,3]")
    gt_ext = GroundTruth("spectral", t_max=3.0)   # solver accepts t_max
    save_pickle(gt_ext, GT_EXT)

ref_fn = lambda P: gt_ext(P[:, 0], P[:, 1])       # (x,t) -> u

stamp("Layer 4 — domain generalisation")
dg = domain.domain_generalization(
    {"qapinn": probe, "classical": classical}, ref_fn, BOUNDS,
    extend_axis=1, factors=(1.0, 1.5, 2.0, 3.0), n=4000, outdir=OUT)
print("  rel-L2 by extension factor:")
for name, errs in dg["rel_l2"].items():
    print(f"    {name}: " + "  ".join(f"{f}×={e:.2e}" for f, e in zip(dg["factors"], errs)))
save_pickle(dg, "layer4_domain.pkl")
stamp("Cell 12 done — Layer 4 complete")

[12:22:42] solving spectral GT on extended window t∈[0,3]
  saved ground_truth_ext.pkl
[12:23:01] Layer 4 — domain generalisation
  rel-L2 by extension factor:
    qapinn: 1.0×=5.43e-01  1.5×=6.56e-01  2.0×=8.37e-01  3.0×=1.36e+00
    classical: 1.0×=4.30e-01  1.5×=4.51e-01  2.0×=4.99e-01  3.0×=7.54e-01
  saved layer4_domain.pkl
[12:23:01] Cell 12 done — Layer 4 complete
